# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

_Dataset Citation: Kulka, M., Rodriguez Miera, S., and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers._

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs by inspecting the Croissant schema. 

**All references use the entity `@id` fields for precise selection.**

In [ ]:
# List all record sets (by @id) and their fields
all_record_sets = metadata.record_sets
print(f"Number of record sets: {len(all_record_sets)}")
for rs in all_record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    print(f"  Description: {rs.get('description', '(no description)')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    for field in fields:
        # field can be a @id or dict depending on Croissant expansion
        fid = field['@id'] if isinstance(field, dict) else field
        print(f"    - {fid}")
    print("---")

## 2.1 Example Record Display
Let's print several example records from one record set. 

Below, replace `<record_set_id>` with the desired `@id` from the output above (for demonstration we use a guessed `cr:ExperimentProteinAbundanceSet` as a plausible main data table, but the real `@id` should be supplied from the output above).

In [ ]:
# Choose the main record set (edit as appropriate based on the previous cell's output)
primary_record_set = None
for rs in all_record_sets:
    if 'protein' in rs.get('name', '').lower() or 'abundance' in rs.get('name', '').lower():
        primary_record_set = rs['@id']
        break
if primary_record_set is None:
    # Just select the first one as fallback
    primary_record_set = all_record_sets[0]['@id']
print(f"Using Record Set @id: {primary_record_set}")

# Print sample records from this record set
for i, record in zip(range(5), dataset.records(record_set=primary_record_set)):
    print(json.dumps(record, indent=2))

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis using the record set and field `@id`s from the overview above.

_You can further extend this cell to extract additional record sets as needed._

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in all_record_sets]

# Extract each record set as a DataFrame
dataframes = {}
for rid in record_set_ids:
    print(f"Extracting {rid} ...")
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rid)))
        dataframes[rid] = df
        print(f" {rid}: shape={df.shape}, columns={list(df.columns)}")
    except Exception as e:
        print(f"  Failed to load {rid}: {str(e)}")

# Inspection: show the first few columns of the main record set
main_df = dataframes[primary_record_set]
print(f"Columns in main record set ({primary_record_set}):\n{main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric criterion, normalizing numeric columns, and grouping data by an attribute. All entity references use field `@id` values.

You may need to adjust `numeric_field_id` and `group_field_id` based on the output of the main data's columns above.

In [ ]:
# Configure the numeric field and group field by their @id from the schema
columns = list(main_df.columns)
print("All main_df columns:", columns)

# Guess likely fields for demonstration (replace with actual @ids as appropriate):
# E.g., '@id': 'cr:protein_abundance', '@id': 'cr:coverage', etc.
possible_numeric = [col for col in columns if any(s in col.lower() for s in ['abundance', 'count', 'coverage', 'mw', 'number'])]
print("Possible numeric fields:", possible_numeric)
if possible_numeric:
    numeric_field_id = possible_numeric[0]  # Pick the first plausible numeric field
else:
    numeric_field_id = columns[0]
print(f"Numeric field selected: {numeric_field_id}")

# Try to find a grouping field (most likely to be something like cr:sample, cr:condition, etc.)
possible_group = [col for col in columns if any(s in col.lower() for s in ['sample', 'group', 'condition'])]
if possible_group:
    group_field_id = possible_group[0]
else:
    group_field_id = None
print(f"Group field selected: {group_field_id}")

# Replace non-numeric with NaN then apply filters
df_numeric = main_df.copy()
df_numeric[numeric_field_id] = pd.to_numeric(df_numeric[numeric_field_id], errors='coerce')

threshold = df_numeric[numeric_field_id].mean() if df_numeric[numeric_field_id].notnull().any() else 0
filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
print(f"Filtered to {len(filtered_df)} rows with {numeric_field_id} > {threshold:.3f}")
print(filtered_df.head())

# Normalize that field
if filtered_df[numeric_field_id].std() > 0:
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
else:
    filtered_df[f"{numeric_field_id}_normalized"] = filtered_df[numeric_field_id]
print("Normalized field:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional groupby
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"Grouped means by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Use matplotlib/seaborn for plotting
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-darkgrid')

# Histogram of the chosen numeric field
plt.figure(figsize=(7, 4))
filtered_df[numeric_field_id].hist(bins=30)
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.title(f"Distribution of {numeric_field_id} (> {threshold:.2f})")
plt.show()

# Grouped barplot if group field is present
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(9, 5))
    sns.barplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the Mass Spectrometry EVs dataset using `mlcroissant`, explored its metadata, explored main record sets by `@id`, filtered and analyzed abundance-related fields, and visualized data distributions. 

For your own applications, refine field selection by inspecting column names and `@id`s, and extend EDA and visualizations as needed.

_Dataset source: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)_